# Next Priority Runs — Exp43 / VS r32

Outputs are Drive-only. This notebook does not update `PROGRESS.md` and does not push result CSVs to GitHub.

Preset order:
- `p0_exp43_s24`: Exp43 C0/C1 on S24, VS init r=32
- `p1_exp43_s01`: Exp43 C0/C1 on S01, VS init r=32
- `p2_vs_s28`: S28 r=32 SD LoRA VS generation
- `p3_vs_s02`: S02 r=32 SD LoRA VS generation


In [ ]:
# [1] GPU check
import torch
assert torch.cuda.is_available(), 'GPU 없음: Runtime > Change runtime type > GPU'
print('torch', torch.__version__)
print('GPU  ', torch.cuda.get_device_name(0))


In [ ]:
# [2] Mount Drive and clone/pull repo
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
REPO = 'https://github.com/heegyukim4043/test_diffusion_EEG_visualstim_image.git'
WORKDIR = '/content/vsvi_project'

if os.path.exists(WORKDIR) and os.path.exists(os.path.join(WORKDIR, '.git')):
    subprocess.run(['git', '-C', WORKDIR, 'pull', 'origin', 'main'], check=True)
else:
    subprocess.run(['rm', '-rf', WORKDIR], check=True)
    subprocess.run(['git', 'clone', REPO, WORKDIR], check=True)

os.chdir(WORKDIR)
subprocess.run(['git', 'log', '--oneline', '-5'], check=False)


In [ ]:
# [3] Select preset and launch background run
# Change PRESET one at a time. Do not run multiple presets simultaneously on one GPU.
PRESET = 'p1_exp43_s01' # 'p0_exp43_s24'  # p0_exp43_s24 | p1_exp43_s01 | p2_vs_s28 | p3_vs_s02

# Exp43 default uses VI/class=60 => 432 train trials. Set FULL_VI=True for all VI trials.
FULL_VI = False

cmd = f"python -u launch_next_priority_runs.py --preset {PRESET}"
if FULL_VI:
    cmd += ' --full_vi'

print(cmd)
!cd /content/vsvi_project && {cmd}


In [ ]:
# [4] Monitor active process and GPU
!pgrep -af 'train_exp43_vi_lora.py|train_vs_re_lora_gen.py' || true
!nvidia-smi
!ls -lt /content/drive/MyDrive/vsvi_data/logs | head -20


In [ ]:
# [5] Tail latest log for the selected preset
import glob, os
patterns = {
    'p0_exp43_s24': '/content/drive/MyDrive/vsvi_data/logs/exp43_s24_c0c1_r32_tok16_*.log',
    'p1_exp43_s01': '/content/drive/MyDrive/vsvi_data/logs/exp43_s01_c0c1_r32_tok16_*.log',
    'p2_vs_s28': '/content/drive/MyDrive/vsvi_data/logs/vs_s28_lora_r32_tok16_*.log',
    'p3_vs_s02': '/content/drive/MyDrive/vsvi_data/logs/vs_s02_lora_r32_tok16_*.log',
}
logs = sorted(glob.glob(patterns[PRESET]), key=os.path.getmtime, reverse=True)
assert logs, f'No log found for {PRESET}'
LOG = logs[0]
print('LOG=', LOG)
!tail -n 120 "{LOG}"


In [ ]:
# [6] Find Drive-only result CSVs
!echo '=== Exp43 VI results ==='
!find /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora -name 'results_exp43_vi_lora.csv' -ls 2>/dev/null | tail -30
!echo '=== VS results ==='
!find /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen -name 'results_lora_gen.csv' -ls 2>/dev/null | tail -30


In [10]:
import os, shutil
if os.path.exists('/content/drive'):
    shutil.rmtree('/content/drive')
from google.colab import drive
drive.mount('/content/drive')

OSError: [Errno 125] Operation canceled: '/content/drive/.Encrypted/.shortcut-targets-by-id'

In [ ]:
%%bash
ls /content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_01_1.npz && echo "✅ Drive OK" || echo "❌ Drive 문제"

In [ ]:
%%bash
# 환경 준비
cd /content
git clone https://github.com/heegyukim4043/test_diffusion_EEG_visualstim_image.git vsvi_project 2>/dev/null
cd /content/vsvi_project && git pull
ln -sfn /content/drive/MyDrive/vsvi_data/preproc_vi_re /content/vsvi_project/preproc_vi_re
pip uninstall -y torchao 2>/dev/null

# 확인
ls /content/vsvi_project/preproc_vi_re/preproc_subj_01_1.npz && echo "✅ Data OK" || echo "❌ Data missing"

# foreground 학습 시작
python -u train_exp43_vi_lora.py \
  --subject_ids 1 \
  --conditions c0,c1 \
  --data_root /content/vsvi_project/preproc_vi_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/vsvi_project/checkpoints_vsre_dino/20260530_095045_ch32_merged_ep200_supcon \
  --init_lora_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260617_074245_lora_r32_ep100/subj01_lora_best.pt \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --per_class_total 135 \
  --eval_n_samples 54 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/exp43_s01_c0c1_r32_tok16_full_$(date +%Y%m%d_%H%M%S).log